# LỰA CHỌN MÔ HÌNH VÀ HUẤN LUYỆN MÔ HÌNH HỌC KẾT HỢP (ENSEMBLE LEARNING)
Notebook này thực hiện quá trình huấn luyện, đánh giá, so sánh các mô hình học máy đơn lẻ và mô hình học kết hợp để tìm ra bộ phân loại tối ưu nhất cho bài toán phân loại thể trạng dinh dưỡng trẻ em.

### Định hướng thực hiện:
1. Tải tập dữ liệu đã qua tiền xử lý (`nhanes_under24_processed.csv`).
2. Thử nghiệm trên **10 thuật toán học máy đơn lẻ** khác nhau.
3. Chọn ra **3 mô hình đơn lẻ tốt nhất (Top 3 Single Models)**.
4. Xây dựng **2 phương pháp học kết hợp (Ensemble Learning)** từ bộ 3 mô hình tốt nhất ở trên:
   * **Voting Classifier** (Bầu chọn mềm - Soft Voting)
   * **Stacking Classifier** (Phân tầng xếp chồng với Logistic Regression làm Meta-Model)
5. So sánh toàn diện hiệu năng của **5 mô hình** (3 mô hình đơn + 2 mô hình kết hợp) trên tập kiểm thử.
6. Chọn mô hình xuất sắc nhất, train lại trên toàn bộ tập dữ liệu, trực quan hóa kết quả và lưu trữ mô hình.

## Bước 1: Kết nối Google Drive và Nạp tập dữ liệu

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Đường dẫn thư mục chính lưu trữ project trên Drive của bạn
# Đồng bộ hóa với File 1 để sử dụng chung một thư mục DACN
DRIVE_BASE_DIR = '/content/drive/MyDrive/DACN'
PROCESSED_CSV = os.path.join(DRIVE_BASE_DIR, 'data/processed/nhanes_under24_processed.csv')
MODEL_OUTPUT_PATH = os.path.join(DRIVE_BASE_DIR, 'models/growth_model_final.joblib')

os.makedirs(os.path.join(DRIVE_BASE_DIR, 'models'), exist_ok=True)

print(f"Đường dẫn file dữ liệu đầu vào: {PROCESSED_CSV}")
print(f"Đường dẫn file model đầu ra: {MODEL_OUTPUT_PATH}")

## Bước 2: Cài đặt và Import các thư viện học máy

In [ ]:
# Cài đặt CatBoost và LightGBM/XGBoost nếu chưa có sẵn
!pip install -q catboost lightgbm xgboost scikit-learn matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

# Import các mô hình đơn lẻ
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Import mô hình học kết hợp
from sklearn.ensemble import VotingClassifier, StackingClassifier

print("Tất cả các thư viện ML đã được import thành công!")

## Bước 3: Đọc và khám phá sơ bộ dữ liệu

In [ ]:
# Đọc file CSV
df = pd.read_csv(PROCESSED_CSV)
print(f"Tải thành công dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head())

# Kiểm tra tỷ lệ nhãn đầu ra để xác định độ cân bằng
plt.figure(figsize=(8, 4))
sns.countplot(x='who_class', data=df, palette='Set2')
plt.title("Phân phối các nhãn thể trạng dinh dưỡng (WHO Class)")
plt.xticks(rotation=45)
plt.show()

## Bước 4: Mã hóa Nhãn (Label Encoding) và Chia tập Train / Test

In [ ]:
# Tách biệt ma trận đặc trưng X và vector mục tiêu y
X = df.drop(columns=['who_class'])
y = df['who_class']

# Điền giá trị khuyết thiếu bằng median, với cột toàn NaN (như head_circumference_cm) điền bằng 0.0 để tránh lỗi NaN
X = X.fillna(X.median()).fillna(0.0)

# Mã hóa nhãn dạng text (ví dụ: normal, thin...) thành dạng số (0, 1, 2...)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Bản đồ ánh xạ nhãn:")
for idx, cls_name in enumerate(label_encoder.classes_):
    print(f"- {idx} ánh xạ từ: {cls_name}")

# Phân chia tập dữ liệu thành Train (80%) và Test (20%) theo cơ chế giữ nguyên tỷ lệ nhãn (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

print(f"\nKích thước tập huấn luyện (Train): {X_train.shape}")
print(f"Kích thước tập kiểm thử (Test): {X_test.shape}")

## Bước 5: Đánh giá Hiệu năng 10 Thuật toán Mô hình Đơn lẻ sử dụng K-Fold

In [ ]:
# Khởi tạo danh sách 10 mô hình học máy đơn lẻ phổ biến
single_models = {
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss'),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM (RBF)": SVC(probability=True, random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

# Khai báo bộ chia K-Fold kiểm thử (Stratified 5-Fold để đảm bảo độ uy tín khoa học)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
evaluation_results = []

print("Tiến hành huấn luyện và đánh giá 10 mô hình đơn lẻ bằng Stratified 5-Fold Cross Validation...")
for name, clf in single_models.items():
    try:
        # Thực hiện đánh giá chéo trên toàn bộ tập dữ liệu X, y_encoded
        # Sử dụng n_jobs=-1 để tận dụng tối đa số luồng CPU xử lý song song
        scores = cross_validate(
            clf, X, y_encoded, 
            cv=cv, 
            scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
            n_jobs=-1
        )
        
        mean_accuracy = np.mean(scores['test_accuracy'])
        std_accuracy = np.std(scores['test_accuracy'])
        mean_precision = np.mean(scores['test_precision_macro'])
        mean_recall = np.mean(scores['test_recall_macro'])
        mean_f1 = np.mean(scores['test_f1_macro'])
        
        # Huấn luyện mô hình đơn lẻ trên tập train cố định để dùng làm Base Estimator ở các bước sau
        clf.fit(X_train, y_train)
        
        evaluation_results.append({
            "Model": name,
            "Accuracy": mean_accuracy,
            "Accuracy_Std": std_accuracy,
            "Precision": mean_precision,
            "Recall": mean_recall,
            "F1-Score": mean_f1,
            "raw_estimator": clf
        })
        print(f"+ {name:20}: Mean Accuracy = {mean_accuracy:.4f} ± {std_accuracy:.4f} | Mean F1 = {mean_f1:.4f}")
    except Exception as e:
        print(f"Xảy ra lỗi khi huấn luyện mô hình {name}: {e}")

# Chuyển kết quả sang DataFrame và sắp xếp giảm dần theo chỉ số Accuracy
evaluation_df = pd.DataFrame(evaluation_results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("\nBảng xếp hạng hiệu năng 10 mô hình đơn lẻ qua 5-Fold Cross Validation:")
display(evaluation_df.drop(columns=["raw_estimator"]))

# Vẽ biểu đồ so sánh độ chính xác trung bình
plt.figure(figsize=(10, 5))
sns.barplot(x='Accuracy', y='Model', data=evaluation_df, palette='Blues_r')
plt.title("So sánh độ chính xác trung bình (Mean Accuracy) qua 5-Fold Cross Validation")
plt.xlabel("Độ chính xác (Accuracy)")
plt.xlim(0.8, 1.0)
plt.show()

## Bước 6: Trích xuất Top 3 Mô hình Đơn lẻ Tốt nhất làm Base Estimators

In [ ]:
# Lấy 3 dòng đầu tiên đại diện cho 3 mô hình tốt nhất
top_3_models = evaluation_df.head(3)
print("3 mô hình đơn tốt nhất được chọn làm Base Estimators cho Ensemble Learning:")

base_estimators = []
for i, row in top_3_models.iterrows():
    model_name = row['Model']
    print(f"#{i+1}: {model_name} (Accuracy: {row['Accuracy']:.4f})")
    
    # Khởi tạo instance mới cho các mô hình tốt nhất để huấn luyện kết hợp
    if model_name == "LightGBM":
        base_estimators.append(('lgbm', LGBMClassifier(random_state=42, verbose=-1)))
    elif model_name == "XGBoost":
        base_estimators.append(('xgb', XGBClassifier(random_state=42, eval_metric='mlogloss')))
    elif model_name == "CatBoost":
        base_estimators.append(('cat', CatBoostClassifier(random_state=42, verbose=0)))
    elif model_name == "Random Forest":
        base_estimators.append(('rf', RandomForestClassifier(random_state=42)))
    elif model_name == "Extra Trees":
        base_estimators.append(('et', ExtraTreesClassifier(random_state=42)))
    elif model_name == "Gradient Boosting":
        base_estimators.append(('gb', GradientBoostingClassifier(random_state=42)))
    elif model_name == "SVM (RBF)":
        base_estimators.append(('svc', SVC(probability=True, random_state=42)))
    elif model_name == "Logistic Regression":
        base_estimators.append(('lr', LogisticRegression(random_state=42, max_iter=1000)))
    elif model_name == "Decision Tree":
        base_estimators.append(('dt', DecisionTreeClassifier(random_state=42)))
    elif model_name == "K-Nearest Neighbors":
        base_estimators.append(('knn', KNeighborsClassifier()))

## Bước 7: Xây dựng và Huấn luyện các mô hình học kết hợp (Ensemble Learning)

In [ ]:
print("Đang xây dựng mô hình Ensemble 1: Voting Classifier (Soft Voting)... ")
voting_classifier = VotingClassifier(estimators=base_estimators, voting='soft')
voting_classifier.fit(X_train, y_train)
print("Voting Classifier đã huấn luyện xong!")

print("\nĐang xây dựng mô hình Ensemble 2: Stacking Classifier (Meta model = Logistic Regression)... ")
stacking_classifier = StackingClassifier(
    estimators=base_estimators, 
    final_estimator=LogisticRegression(max_iter=1000)
)
stacking_classifier.fit(X_train, y_train)
print("Stacking Classifier đã huấn luyện xong!")

## Bước 8: So sánh toàn diện 5 mô hình (3 đơn lẻ + 2 kết hợp)

In [ ]:
final_comparison_results = []

# 1. Thêm kết quả của 3 mô hình đơn tốt nhất đã lưu
for idx, row in top_3_models.iterrows():
    final_comparison_results.append({
        "Model_Type": "Mô hình đơn lẻ",
        "Model_Name": row['Model'],
        "Accuracy": row['Accuracy'],
        "Precision": row['Precision'],
        "Recall": row['Recall'],
        "F1-Score": row['F1-Score'],
        "estimator_instance": row['raw_estimator']
    })

# 2. Đánh giá mô hình Voting kết hợp trên tập test
v_preds = voting_classifier.predict(X_test)
v_acc = accuracy_score(y_test, v_preds)
v_prec, v_rec, v_f1, _ = precision_recall_fscore_support(y_test, v_preds, average='macro')
final_comparison_results.append({
    "Model_Type": "Học kết hợp (Ensemble)",
    "Model_Name": "Voting Classifier (Soft)",
    "Accuracy": v_acc,
    "Precision": v_prec,
    "Recall": v_rec,
    "F1-Score": v_f1,
    "estimator_instance": voting_classifier
})

# 3. Đánh giá mô hình Stacking kết hợp trên tập test
s_preds = stacking_classifier.predict(X_test)
s_acc = accuracy_score(y_test, s_preds)
s_prec, s_rec, s_f1, _ = precision_recall_fscore_support(y_test, s_preds, average='macro')
final_comparison_results.append({
    "Model_Type": "Học kết hợp (Ensemble)",
    "Model_Name": "Stacking Classifier (LR Meta)",
    "Accuracy": s_acc,
    "Precision": s_prec,
    "Recall": s_rec,
    "F1-Score": s_f1,
    "estimator_instance": stacking_classifier
})

# Chuyển sang DataFrame và sắp xếp để xếp hạng
final_comparison_df = pd.DataFrame(final_comparison_results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("Bảng xếp hạng so sánh 5 mô hình cuối cùng:")
display(final_comparison_df.drop(columns=["estimator_instance"]))

# Trực quan hóa so sánh độ chính xác
plt.figure(figsize=(10, 5))
sns.barplot(x='Accuracy', y='Model_Name', hue='Model_Type', data=final_comparison_df, palette='Set1')
plt.title("So sánh độ chính xác giữa các mô hình đơn lẻ và học kết hợp (Ensemble)")
plt.xlabel("Độ chính xác (Accuracy)")
plt.xlim(0.85, 1.0)
plt.legend(loc='lower right')
plt.show()

## Bước 9: Huấn luyện lại Model Tốt nhất trên Toàn bộ Tập Dữ liệu và Đánh giá chi tiết

In [ ]:
# Chọn ra model tốt nhất đứng đầu bảng xếp hạng
best_row = final_comparison_df.iloc[0]
best_name = best_row['Model_Name']
best_estimator = best_row['estimator_instance']

print(f"Mô hình tốt nhất được chọn: {best_name} với độ chính xác trên tập kiểm thử = {best_row['Accuracy']:.4f}")

# Huấn luyện lại mô hình trên toàn bộ tập dữ liệu (gộp cả train và test để đạt lượng thông tin tối đa)
print(f"\nĐang huấn luyện lại {best_name} trên toàn bộ dữ liệu...")
best_estimator.fit(X, y_encoded)
print("Hoàn tất huấn luyện mô hình cuối cùng!")

# Đánh giá chi tiết
final_predictions = best_estimator.predict(X_test)
print("\nClassification Report trên tập test:")
print(classification_report(y_test, final_predictions, target_names=label_encoder.classes_))

# Trực quan hóa Confusion Matrix
cm = confusion_matrix(y_test, final_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', 
    xticklabels=label_encoder.classes_, 
    yticklabels=label_encoder.classes_
)
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Nhãn dự báo (Predicted Label)")
plt.ylabel("Nhãn thực tế (True Label)")
plt.show()

## Bước 10: Đóng gói và lưu trữ mô hình huấn luyện sang Google Drive

In [ ]:
# Tạo bundle đóng gói mô hình tốt nhất, bộ giải mã nhãn và danh sách thuộc tính đưa vào
model_bundle = {
    "model": best_estimator,
    "label_encoder": label_encoder,
    "feature_columns": list(X.columns)
}

# Serialization lưu sang Drive làm model chạy cho backend chính của ứng dụng di động
joblib.dump(model_bundle, MODEL_OUTPUT_PATH)

print(f"Mô hình đã được lưu thành công tại Google Drive:")
print(f"--> {MODEL_OUTPUT_PATH}")